# 06 — Cost, latency and calibration

Economics of every council condition and confidence calibration of every member. Cost is the provider-reported cost per call (MEASURED); local models report 0. Latency of a council is the slowest member plus the judge where used (ESTIMATED from measured per-call latency: members are assumed to run in parallel). Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


In [3]:
order = ["A_solo:sonnet", "A_solo:haiku", "B1_same_provider_roles", "B2_same_provider_sampling", "C_multi_provider",
         "D_characteristic_diverse", "E_judge", "F_adaptive_routing"]
base = R["conditions"]["A_solo:sonnet"]
econ = pd.DataFrame([{"condition": k, "accuracy": R["conditions"][k]["accuracy"],
                      "calls": R["conditions"][k]["calls_per_decision"], "tokens": R["conditions"][k]["tokens_per_decision"],
                      "latency_s": R["conditions"][k]["latency_s_per_decision"], "cost_usd": R["conditions"][k]["cost_usd_per_decision"]}
                     for k in order]).set_index("condition")
econ["tokens_x_solo"] = (econ.tokens / base["tokens_per_decision"]).round(2)
econ["latency_x_solo"] = (econ.latency_s / base["latency_s_per_decision"]).round(2)
econ["cost_x_solo"] = (econ.cost_usd / base["cost_usd_per_decision"]).round(2)
econ.to_csv(EV.parent / "evidence" / "council" / "cost_latency_by_condition.csv")
econ

,accuracy,calls,tokens,latency_s,cost_usd,tokens_x_solo,latency_x_solo,cost_x_solo
condition,,,,,,,,
A_solo:sonnet,0.9859,1.00,431.6,2.31,0.001994,1.00,1.00,1.00
A_solo:haiku,0.9859,1.00,489.8,1.77,0.000129,1.13,0.77,0.06
B1_same_provider_roles,0.9718,3.00,1786.3,2.43,0.000490,4.14,1.05,0.25
B2_same_provider_sampling,0.9718,3.00,1458.6,2.10,0.000381,3.38,0.91,0.19
C_multi_provider,0.9859,3.00,1187.4,2.71,0.002122,2.75,1.17,1.06
D_characteristic_diverse,0.9718,3.00,1344.2,2.72,0.002177,3.11,1.18,1.09
E_judge,0.9859,4.00,1927.0,5.26,0.004899,4.46,2.28,2.46
F_adaptive_routing,0.9718,3.35,1500.6,3.07,0.002029,3.48,1.33,1.02


In [4]:
recorded = pd.concat([pd.DataFrame(jsonl("council/member_calls.jsonl")), pd.DataFrame(jsonl("council/judge_calls.jsonl"))])
calls = final_calls("council/member_calls.jsonl")
print("recorded calls (incl. re-asked):", len(recorded), "| analysed member answers:", len(calls))
allc = recorded
per_member = allc.groupby("member").agg(calls=("case_id", "size"), tokens_in=("tokens_in", "sum"), tokens_out=("tokens_out", "sum"),
                                        cost_usd=("cost_usd", "sum"), latency_p50=("latency_s", "median"),
                                        latency_p95=("latency_s", lambda s: s.quantile(0.95)), retries=("retries", "sum"))
arch = pd.DataFrame(jsonl("council/member_calls_budget300_archive.jsonl"))
# The archive is the pre-Amendment-2 file: its rows are the first rows of the final cache, not extra calls.
assert jsonl("council/member_calls_budget300_archive.jsonl") == jsonl("council/member_calls.jsonl")[:len(arch)]
print("distinct recorded calls:", len(allc), "| provider-reported cost of all of them (USD):",
      round(float(allc.cost_usd.fillna(0).sum()), 4))
print("note: adding the archive again would double-count", len(arch), "calls (USD",
      round(float(arch.cost_usd.fillna(0).sum()), 4), ")")
per_member.round(4)

recorded calls (incl. re-asked): 1506 | analysed member answers: 1210
distinct recorded calls: 1506 | provider-reported cost of all of them (USD): 0.714
note: adding the archive again would double-count 1210 calls (USD 0.3222 )


,calls,tokens_in,tokens_out,cost_usd,latency_p50,latency_p95,retries
member,,,,,,,
haiku,130,37984,27862,0.0177,1.715,3.4465,0
haiku_critic,141,48265,37749,0.0237,1.970,3.4000,0
haiku_domain,131,41952,30791,0.0196,1.820,3.0350,0
haiku_repeat,128,37418,26799,0.0171,1.635,2.8425,0
haiku_s1,126,36996,27326,0.0174,1.640,3.4300,0
haiku_s2,124,36399,25502,0.0164,1.605,2.6040,0
haiku_s3,124,36218,25688,0.0165,1.550,2.6885,0
haiku_verifier,154,54052,45781,0.0283,2.030,3.5450,0
judge,110,63580,18135,0.3085,2.155,4.4570,0


## Calibration (EXPLORATORY — not pre-registered)

Expected calibration error over 10 equal-width confidence bins and Brier score of the stated confidence for the chosen option, on the evaluation split. Computed here from the frozen answers; an abstention counts as wrong with confidence 0.

In [5]:
truth = pd.DataFrame(load("council/cases.json")["cases"]).set_index("case_id")
ev = calls[calls.case_id.map(truth.split) == "evaluation"].copy()
ev["correct"] = (ev.answer == ev.case_id.map(truth.answer)).astype(float)
ev["conf"] = ev.confidence.fillna(0.0).clip(0, 1)


def ece(df, bins=10):
    idx = np.minimum((df.conf * bins).astype(int), bins - 1)
    return float(sum(abs(g.correct.mean() - g.conf.mean()) * len(g) for _, g in df.groupby(idx)) / len(df))


cal = pd.DataFrame([{"member": m, "n": len(g), "accuracy": g.correct.mean(), "mean_confidence": g.conf.mean(),
                     "ece": ece(g), "brier": float(((g.conf - g.correct) ** 2).mean())} for m, g in ev.groupby("member")]).set_index("member")
cal.to_csv(EV / "council" / "calibration_by_member.csv")
cal.round(3)

,n,accuracy,mean_confidence,ece,brier
member,,,,,
haiku,71,0.986,0.857,0.129,0.039
haiku_critic,71,0.986,0.845,0.141,0.036
haiku_domain,71,0.972,0.870,0.109,0.037
haiku_repeat,71,0.986,0.858,0.128,0.036
haiku_s1,71,0.972,0.859,0.113,0.039
haiku_s2,71,0.986,0.856,0.129,0.038
haiku_s3,71,0.958,0.862,0.102,0.045
haiku_verifier,71,0.958,0.843,0.115,0.049
qwen1_5b,71,0.282,0.863,0.581,0.535


**RESULT.** Always-on councils cost 3–4 calls, 2.7–4.5× tokens, up to 2.3× latency (judge) and up to 2.5× provider cost of the strong solo model for no accuracy gain on this set. A solo Haiku matched the strong solo accuracy at about 1/15 of its cost per decision. Calibration figures are exploratory and describe this case set only.

**UNKNOWN:** cost of Jev calls (no price reported), latency on a production network path, prompt-cache effects (Hermes-style stable prefix — `NOT_RUN`).